# Fall Detection & Posture Classification — Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pat2echo/AI-Posture-Monitor/blob/master/notebooks/fall_detection_posture_classification_demo.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/patrickogbuitepu/fall-detection-posture-classification-starter)

Demo of the `ai-posture-monitor` package (MediaPipe pose estimation + fuzzy-logic rules + a finite state machine) on the [Fall Detection and Posture Classification Dataset](https://www.kaggle.com/datasets/patrickogbuitepu/posture-monitor-and-fall-detection).

Runs unmodified on **Kaggle**, **Google Colab**, or locally — [`kagglehub`](https://github.com/Kaggle/kagglehub) detects the environment automatically: on Kaggle it reads the already-mounted dataset with no network call, on Colab/local it downloads only the handful of small files this notebook actually needs (no large videos).

This notebook:
1. Runs the real `ai-posture-monitor` package on sample images straight from the published Kaggle dataset
2. Explores the dataset's activity/fall event labels
3. Checks the dataset's own precomputed static-posture classification accuracy
4. Visualizes precomputed per-frame posture output on a training video

Source code: [pat2echo/AI-Posture-Monitor](https://github.com/pat2echo/AI-Posture-Monitor)

**Changelog:** earlier releases of `ai-posture-monitor` raised a `SyntaxError` on import under Python <3.12 ([pose_est_dependencies.py#L337](https://github.com/pat2echo/AI-Posture-Monitor/blob/master/ai_posture_monitor_package/ai_posture_monitor/pose_est_dependencies.py#L337) nested same-type quotes inside an f-string, only legal since PEP 701/Python 3.12) and forced the `TkAgg` matplotlib backend at import time, which fails on headless notebook environments. Both are fixed as of package version `0.0.17`.

In [ ]:
!pip install -q kagglehub "ai-posture-monitor==0.0.17"

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import ai_posture_monitor as pm

DATASET = 'patrickogbuitepu/posture-monitor-and-fall-detection'


def kaggle_file(relative_path, retries=3):
    """Resolve one file from the dataset.

    kagglehub detects the runtime environment: on a Kaggle kernel it reads the
    dataset already mounted at /kaggle/input with no network call; on Colab or
    locally it downloads (and caches) just this one file via the Kaggle API.
    """
    last_err = None
    for _ in range(retries):
        try:
            return kagglehub.dataset_download(DATASET, path=relative_path)
        except Exception as e:
            last_err = e
    raise last_err

## 1. Live Pose Estimation on Kaggle Sample Images

Run the actual package on three sample static-pose images straight from the dataset — `train/pose/stand.jpg`, `sit.jpg`, `lie.jpg` — and compare the package's rule-based classification against the ground truth implied by each filename.

Note: MediaPipe's landmark detector is run here at its default confidence threshold and can fail to return any landmarks at all on some lying-down poses (self-occlusion, unusual silhouette from a fixed camera angle) — this is precisely why the dataset also ships bounding-box-aspect-ratio-based features (`static_pose_boundingbox_data*.csv`) as a fallback signal alongside the landmark-based ones. The cell below handles that case rather than failing.

In [ ]:
pose, mp_drawing, mp_pose = pm.initialize_mediapipe()
sample_images = ['stand.jpg', 'sit.jpg', 'lie.jpg']
feature_cols = pm.get_attr_of_features()

fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 5))
rows = []

for ax, name in zip(axes, sample_images):
    img_path = kaggle_file(f'train/pose/{name}')
    results, img_rgb, landmarks_df = pm.detect_pose_landmarks(img_path, pose=pose, show=True)

    annotated = img_rgb.copy()
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(annotated, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        landmarks_3d = landmarks_df[['X', 'Y', 'Z']].to_numpy()
        feats = pm.get_features(landmarks_3d, image_name=name)
    else:
        feats = [name] + [None] * (len(feature_cols) - 1)

    ax.imshow(annotated)
    ax.set_title(name if results.pose_landmarks else f'{name}\n(no landmarks detected)')
    ax.axis('off')
    rows.append(feats)

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(rows, columns=feature_cols)
results_df.insert(1, 'expected_pose', [n.split('.')[0] for n in sample_images])
results_df[['image_name', 'expected_pose', 'is_upright', 'stand_left', 'stand_right', 'sit_left', 'sit_right', 'lie_left', 'lie_right']]

## 2. Activity & Fall Event Labels

Every video has a matching label CSV with columns `start_time`, `end_time`, `action`, `is_fall` (seconds from the start of the video).

In [ ]:
train_labels = [f'hr_fall_detection_{i}' for i in (1, 2, 3)]
valid_labels = [f'fall_detection_{i}' for i in range(4, 11)]

frames = []
for name in train_labels:
    path = kaggle_file(f'train/labels/{name}.csv')
    df = pd.read_csv(path)
    df['video'] = name
    frames.append(df)
for name in valid_labels:
    path = kaggle_file(f'valid/labels/{name}.csv')
    df = pd.read_csv(path)
    df['video'] = name
    frames.append(df)

labels_df = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(frames)} label files, {len(labels_df)} segments total')
labels_df.head()

In [ ]:
action_counts = labels_df['action'].fillna('None').value_counts()

plt.figure(figsize=(9, 4))
action_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Labelled activity segments across all videos')
plt.ylabel('Number of segments')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

fall_counts = labels_df[labels_df['is_fall'] == True].groupby('video').size()

plt.figure(figsize=(9, 4))
fall_counts.plot(kind='bar', color='indianred', edgecolor='black')
plt.title('Labelled fall segments per video')
plt.ylabel('Fall segments')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Static Posture Classification Accuracy (precomputed)

The dataset ships `features_output_predicted.csv` for the 113 static pose images — MediaPipe-derived features plus a `label` (ground truth from the filename) and `predicted_label` (the package's rule-based classifier).

In [ ]:
features_path = kaggle_file('train/pose/features_output_predicted.csv')
features_df = pd.read_csv(features_path)
print(features_df.shape)
features_df[['image_name', 'label', 'predicted_label']].head(10)

In [ ]:
accuracy = (features_df['label'] == features_df['predicted_label']).mean()
print(f'Rule-based static posture classification accuracy on {len(features_df)} images: {accuracy:.1%}')

pd.crosstab(features_df['label'], features_df['predicted_label'])

## 4. Per-Frame Posture Output on a Training Video (precomputed)

`train/results of static pose classifier on training videos/` has one results CSV per training video with per-frame `label` (ground truth) and `prediction` (the package's output) columns — already computed, so we can visualize the system's actual output without re-running the full video pipeline.

In [ ]:
results_path = kaggle_file(
    'train/results of static pose classifier on training videos/'
    'hr_fall_detection_static_pose_class_on_video_1_results.csv'
)
results_df = pd.read_csv(results_path)
print(results_df.shape)
print('Columns:', results_df.columns.tolist())
results_df.head()

In [ ]:
priority = {'stand': 3, 'sit': 2, 'lie': 1}


def base_state(x):
    if pd.isna(x):
        return None
    return str(x).split('-')[0].lower()


prediction_col = next((c for c in ('smooth_prediction', 'prediction') if c in results_df.columns), None)

plot_df = results_df.copy()
plot_df['label_y'] = plot_df['label'].apply(base_state).map(priority)

plt.figure(figsize=(14, 4))
plt.step(plot_df.index, plot_df['label_y'], where='post', label='Ground truth', linewidth=2)

if prediction_col:
    plot_df['pred_y'] = plot_df[prediction_col].apply(base_state).map(priority)
    plt.step(plot_df.index, plot_df['pred_y'], where='post', label=f'Prediction ({prediction_col})', linewidth=2, alpha=0.7)
else:
    print('No prediction column found in this file - plotting ground truth only.')

plt.yticks(list(priority.values()), list(priority.keys()))
plt.xlabel('Frame')
plt.title(f'Static posture: ground truth vs prediction\n{results_path.split("/")[-1]}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fall_cols = [c for c in ('fall_watch', 'fall') if c in results_df.columns]

if fall_cols:
    plt.figure(figsize=(14, 3))
    for c in fall_cols:
        plt.step(plot_df.index, results_df[c].astype(float), where='post', label=c, linewidth=2)
    plt.xlabel('Frame')
    plt.yticks([0, 1], ['No', 'Yes'])
    plt.title('Fall detection output')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('This results file is from a pipeline run that only covers static-pose classification')
    print('(no fall/fall_watch columns). is_fall counts from the matching label file instead:')
    print(labels_df[labels_df['video'] == 'hr_fall_detection_1']['is_fall'].value_counts())

## Citation

```
Ogbuitepu, P. O. AI-Driven Posture Analysis Fall Detection System for the Elderly: Dataset [Data set].
University of Essex.
```

Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Package and full pipeline source: [pat2echo/AI-Posture-Monitor](https://github.com/pat2echo/AI-Posture-Monitor).